In [1]:
from PIL import Image
import torch
from torch import nn
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode# original https://github.com/mlfoundations/open_clip (not for this code)

In [2]:
import sys, importlib, os

for path in ["script", "..", "../.."]:
    path = os.path.abspath(path)
    if path not in sys.path:
        sys.path.append(path)


In [3]:
import script.custom_models.panderm_custom_classifier
import importlib
importlib.reload(script.custom_models.panderm_custom_classifier)
from geom_hyp import lorentz as L
importlib.reload(L)
from geom_hyp import manifold_conversion as MC
from script.custom_models.panderm_custom_classifier import PanDermWithHead, PanDermWithHypHead

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.10/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


cuda


In [4]:
# Hyperbolic Network
results = torch.load("../../Datasets/sarker_embeddingDerm12345 datasets_64_small_radius.pt")
z_p = results["data"]  # torch.Tensor
z_l = MC.poincare_to_lorentz(z_p) # lorentz points

# scaling 
mul_factor = 2.0
z_l = L.exp_map0(L.log_map0(z_l[:, 1:]) * mul_factor)

# model = DermLIPWithHypHead(64, l_target= z_l[0:40,1:].to(device), loss_type= 'ce', 
#                            logit_scale = 2, entail_weight= 0.1, multilayer= True, model_type= 'panderm') # model_type = 'clip'


model = PanDermWithHypHead(64, l_target= z_l[0:40,:].to(device), loss_type= 'ce', 
                           logit_scale = 3, entail_weight= 0.1, multilayer= True, model_type= 'base') # MILK10K

# lorentz space components [selecting  the leaf classes]
# model = PanDermWithHypHead(64, l_target= z_l[0:40,1:].to(device), loss_type= 'ce', logit_scale = 3, entail_weight= 0.02) # lorentz space components [selecting  the leaf classes]

/scratch/26550277/ipykernel_3650865/4082701289.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  results = torch.load("../../Datasets/sarker_embeddingDerm12345 datasets_64

In [5]:
!nvidia-smi

Mon Oct 19 13:51:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:17:00.0 Off |                    0 |
| N/A   26C    P0             36W /  250W |     427MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
torch.acosh(model.l_target.pow(2).sum(dim = -1).add(1).sqrt())
torch.acosh(model.l_target.pow(2).sum(dim = -1).add(1).sqrt())
mini = L.pairwise_dist(model.l_target, model.l_target)
x_masked = mini.clone()
x_masked.fill_diagonal_(float("inf"))

min_val = x_masked.min()
idx = torch.argwhere(x_masked == min_val)[0]
print(min_val)
print(idx)  # tensor([row, col])

tensor(1.6746, device='cuda:0')
tensor([ 5, 27], device='cuda:0')


In [ ]:
# Euclidean Baseline
print('check run:Euclidean Model')
model = PanDermWithHead(11)

In [6]:
# Augmentation Scheme Loop + data loading. 

import Datasets.MILK_10K.Milk10_dataset_loader 
import Datasets.isic_kaggle_2016.IsicKaggle2016
import Datasets.fitzpatrick17k.fitzpatrick_data_loader

importlib.reload(Datasets.MILK_10K.Milk10_dataset_loader)

from Datasets.MILK_10K.Milk10_dataset_loader import MILK10kDataset, get_image_transform
from Datasets.isic_kaggle_2016.IsicKaggle2016 import TumorDataset
from Datasets.fitzpatrick17k.fitzpatrick_data_loader import Fitz17kDataset

import Datasets.MILK_10K.augmentation_scheme_image
import importlib
importlib.reload(Datasets.MILK_10K.augmentation_scheme_image)

from Datasets.MILK_10K.augmentation_scheme_image import _augmentation_gpu

# data loading. 

import sys, importlib
path_to_root = os.path.abspath(os.path.join('..'))
if path_to_root not in sys.path:
    sys.path.append(path_to_root)




# Derm12345 loading

path_to_root = os.path.abspath(os.path.join('../../../../../../projects/academic/sangst/zahidhas/Datasets'))
if path_to_root not in sys.path:
    sys.path.append(path_to_root)

from DERM12345.derm12345_data_loader import DERM12345Dataset


# ISIC loading
from isic_2019.isic_2016_dataset_loader import ISIC2019Dataset

In [7]:
im_size = 224

transform_train_no_np = transforms.Compose([
    transforms.Resize((im_size, im_size),  interpolation=InterpolationMode.BICUBIC),  # Resize the image
    transforms.PILToTensor()  # Convert to uint8 tensor array [c, h, w]
])

constants_zoo = {
    'imagenet': {'mean': (0.485, 0.456, 0.406), 'std': (0.228, 0.224, 0.225)},
    'openai_clip':{'mean': (0.48145466, 0.4578275, 0.40821073), 'std': (0.26862954, 0.26130258, 0.27577711)},
    'uniform': {'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5)}
}
constants = constants_zoo['openai_clip']

mean, std = constants.get('mean'), constants.get('std')
dl_preprocess = transforms.Compose([
    transforms.Normalize(mean=mean, 
                         std=std),
])

In [8]:
# MILK 10K dataset
dataset_keep_track = "MILK 10K datasets"
print(dataset_keep_track)

root             = '../../Datasets/MILK_10K/Train'
split            = 'train'

train_dataset =  MILK10kDataset(
        root             = root, 
        split            = split,
        image_transform  = get_image_transform(split, im_size),
        balanced         = False,
        seed             = 44,
    )


MILK 10K datasets


In [8]:
# ISIC 2019 datasets
root = "../../../../../../projects/academic/sangst/zahidhas/Datasets/isic_2019"

dataset_keep_track = "Isic2019 datasets"

img_size = 224

train_dataset = ISIC2019Dataset(
    root=root,
    split="train",
    image_transform=get_image_transform('train', img_size=img_size),
    balanced=False,
    seed=44,
)

In [8]:
# Derm12345 dataset
dataset_keep_track = "Derm12345 datasets"
print(dataset_keep_track)

root             = "../../../../../../projects/academic/sangst/zahidhas/Datasets/DERM12345/"

img_size = 224

train_dataset  = DERM12345Dataset(
        root=root,
        split="train",
        image_transform=get_image_transform("train", img_size = img_size),
        target_col="label",
        balanced=False,
        seed=44,
    )

Derm12345 datasets


In [18]:
from torch.utils.data import DataLoader

print(dataset_keep_track)
train_loader = DataLoader(
    train_dataset,
    batch_size= 160,
    shuffle=True,
    num_workers=0, 
    persistent_workers=False,
)

Derm12345 datasets


In [11]:
# for name, value in vars(model_2).items():
#         print(f"{name}: {value}")

model = model.to(device)

# Define parameter groups
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
import math

lr1 = 9e-5   # lower lr for pretrained ViT backbone
lr2 = 8e-4   # higher lr for classification head

param_groups = []
all_params = list(model.parameters())
added_params = set()

def add_params(mod, lr):
    for p in mod.parameters():
        if p not in added_params:
            param_groups.append({"params": [p], "lr": lr})
            added_params.add(p)
        
# Custom groups
if hasattr(model, "visual"):
    add_params(model.visual, lr1)

if hasattr(model, "clf_head"):
    add_params(model.clf_head, lr2)

if hasattr(model, "projection_head"):
    add_params(model.clf_head, lr2)

# Add any remaining parameters not covered
for p in model.parameters():
    if p not in added_params:
        param_groups.append({"params": [p], "lr": lr1})


# optimizer = optim.AdamW(param_groups, weight_decay=1e-4)  # AdamW is standard for ViT
## or alternatively
optimizer = optim.SGD(param_groups, momentum=0.99, weight_decay=1e-4) # comment the scheduler in this case


# Optional sanity check
covered = sum(p.numel() for g in optimizer.param_groups for p in g["params"])
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Covered parameters: {covered}/{total} ({covered/total:.2%})")

# Optional: list groups and learning rates
# for i, g in enumerate(optimizer.param_groups):
#     print(f"Group {i}: lr={g['lr']}, num_params={len(g['params'])}")

# # Warmup + Cosine Decay Scheduler
# warmup_epochs = 5
# total_epochs = 100

# def lr_lambda(current_epoch):
#     if current_epoch < warmup_epochs:

#         # Linear warmup
#         return current_epoch / max(1, warmup_epochs)
#     else:
#         # Cosine decay after warmup
#         progress = (current_epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
#         return 0.5 * (1.0 + math.cos(math.pi * progress))

# scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

# # In training loop:
# # optimizer.step()
# # scheduler.step()  # call once per epoch

✅ Covered parameters: 86792747/86638889 (100.18%)


In [ ]:
# training Loop

import random
import pdb
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
im_size = 224
scaler = GradScaler()
grad_clip = 5  # gradient clipping range

ignore_index = -100

print("ignoring: ", ignore_index)

# zero_shot = False
# un_l = [3,5,13,19]  # unlabeled class indices

model.train()
total_epochs = 60 # No of Epoch

for epoch in range(total_epochs):  # loop over the dataset multiple times    
    running_loss = 0.0
    # pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{total_epochs}")

    for i, data in enumerate(train_loader, 0):
        # --- Data loading ---
        orig_image, orig_label = data
        
        if dataset_keep_track == "MILK 10K datasets":
            key = random.choice(['dermoscopic', 'clinical_closeup'])
            images = orig_image[key]
            # labels = orig_label['class_vector'].long() # Euclidean
            labels = orig_label['class_index']
            
        elif dataset_keep_track in ["ISIC 2016 datasets", "Fitzpatrick datasets", 'Derm12345 datasets', 'Isic2019 datasets']:
            images = orig_image         # adjust key as needed
            labels = orig_label['class_index'].long()
        else:
            raise ValueError(f"Unknown dataset: {dataset_keep_track}")
            
        images = images.to(device)
        images = _augmentation_gpu(
            images.float(),
            augmentation_num=random.randint(0, 3),
            im_size=im_size)
        
        images = images/255.0
        input_image = dl_preprocess(images)  
        input_image = input_image.to(device)
        labels = labels.to(device)

        if not (labels != ignore_index).any():continue        
        # --- Forward + backward ---
        # --- Forward ---
        # optimizer.zero_grad()
        # with autocast():
        #     outputs = model(input_image, labels, ignore_index = ignore_index, mul_factor = mul_factor)
        #     loss = outputs["loss"]
        # # --- NaN check before backward ---
        # if torch.isnan(loss):
        #     print(f"\n[NaN Detected] Epoch {epoch+1} Step {i}")
        #     print("Logging:", outputs.get("logging", {}))
        #     pdb.set_trace()
        # # --- Backward ---#
        # scaler.scale(loss).backward()
        # scaler.unscale_(optimizer)
        # torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        # scaler.step(optimizer)
        # scaler.update()


                   # Forward + Backward — FP32
        # =====================================================

        optimizer.zero_grad()
        # --- Forward (NO autocast) ---
        outputs = model(input_image,labels,ignore_index=ignore_index,mul_factor=mul_factor)
        loss = outputs["loss"]

        # --- NaN / Inf check before backward ---
        if not torch.isfinite(loss):
            print(f"\n[Non-finite Loss] Epoch {epoch+1} Step {i}")
            print("Loss:", loss.item())
            print("Logging:", outputs.get("logging", {}))
            pdb.set_trace()

        # --- Backward (NO GradScaler) ---
        loss.backward()

        # --- Gradient clipping ---
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip)
        # --- NaN / Inf gradient check ---
        if not torch.isfinite(grad_norm):
            print(f"\n[Non-finite Gradient] Epoch {epoch+1} Step {i}")
            print("Loss:", loss.item())
            print("Grad norm:", grad_norm.item())
            for name, p in model.named_parameters():
                if p.grad is not None and not torch.isfinite(p.grad).all():
                    print("BAD GRAD:", name)
            pdb.set_trace()

        # --- Optimizer step ---
        optimizer.step()

        # --- Logging --- #
        running_loss += loss.item()
        avg_loss = running_loss / (i + 1)
        # pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{avg_loss:.4f}")
        if i % 100 == 0:
            print(f"[Epoch {epoch} | Step {i}] loss: {loss.item():.4f}")
    # scheduler.step() 
    print(f"Epoch {epoch} avg loss: {running_loss / len(train_loader):.4f}")

/scratch/26550277/ipykernel_3650865/2276252657.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


ignoring:  -100
[Epoch 0 | Step 0] loss: 0.1511
Epoch 0 avg loss: 0.3742
[Epoch 1 | Step 0] loss: 0.1363
Epoch 1 avg loss: 0.3585
[Epoch 2 | Step 0] loss: 0.2863
Epoch 2 avg loss: 0.3620
[Epoch 3 | Step 0] loss: 0.6329
Epoch 3 avg loss: 0.3582
[Epoch 4 | Step 0] loss: 0.1096
Epoch 4 avg loss: 0.3700
[Epoch 5 | Step 0] loss: 0.2612
Epoch 5 avg loss: 0.3466
[Epoch 6 | Step 0] loss: 0.1967
Epoch 6 avg loss: 0.3241
[Epoch 7 | Step 0] loss: 0.0909
Epoch 7 avg loss: 0.3035
[Epoch 8 | Step 0] loss: 0.2711
Epoch 8 avg loss: 0.3399
[Epoch 9 | Step 0] loss: 0.7265
Epoch 9 avg loss: 0.3244
[Epoch 10 | Step 0] loss: 0.1406


In [33]:
# with torch.no_grad():
#     vit_oupt = model.visual(input_image)
#     model_oupt = model.clf_head(vit_oupt)
#     #vit_oupt.norm(dim = -1).mean()

In [38]:
# some post analysis, may delete it later
dd = L.pairwise_dist(outputs['output'], model.l_target)
model.logit_scale.exp()
data_sample = 35
print(torch.softmax(2*(-dd), dim = -1)[data_sample])
labels[data_sample]

tensor([0.0055, 0.1636, 0.0394, 0.0439, 0.0311, 0.0185, 0.0684, 0.0905, 0.0270,
        0.4508, 0.0613], device='cuda:0', grad_fn=<SelectBackward0>)


tensor(9, device='cuda:0')

In [12]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")


#ls ../../../../../projects/academic/sangst/zahidhas/Model_Weights/Derm1M/ # weights save directory

PyTorch Version: 2.5.0a0+872d972e41.nv24.08
CUDA Available: True
GPU Model: NVIDIA A100-PCIE-40GB
Device Capability: (8, 0)


In [20]:
# Saving block. 

PATH  = '../../../../../../projects/academic/sangst/zahidhas/Model_Weights/PanDerm/PanDerm_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth'

response = input(
    f"\nCheckpoint path:\n{PATH}\n\n"
    "Type SAVE to save the model, or ABORT to cancel: ").strip()


if response == "SAVE":
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        # "scheduler_state_dict": scheduler.state_dict(),
        "loss": loss,
    }, PATH)

    print(f"✅ Model saved to {PATH}")

elif response == "ABORT":
    print("🛑 Save aborted.")

else:
    print(f"❌ Invalid response '{response}'. Nothing was saved.")


Checkpoint path:
../../../../../../projects/academic/sangst/zahidhas/Model_Weights/PanDerm/PanDerm_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth

Type SAVE to save the model, or ABORT to cancel:  SAVE


✅ Model saved to ../../../../../../projects/academic/sangst/zahidhas/Model_Weights/PanDerm/PanDerm_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth


In [12]:
# loading block

PATH  = '../../../../../../projects/academic/sangst/zahidhas/Model_Weights/PanDerm/PanDerm_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth'

response = input(
    f"\nCheckpoint path:\n{PATH}\n\n"
    "Type LOAD to load the model, or ABORT to cancel: ").strip()



if response == "LOAD":

    checkpoint = torch.load(PATH, map_location=device)

    msg = model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=False
    )

    print("Missing:", msg.missing_keys)
    print("Unexpected:", msg.unexpected_keys)

    # Load optimizer only if you want to RESUME training
    load_optimizer = input(
        "\nType OPTIMIZER to also load optimizer state, "
        "or SKIP to keep the current optimizer: "
    ).strip()

    if load_optimizer == "OPTIMIZER":
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )
        print("✅ Optimizer state loaded.")

    else:
        print("⏭️ Optimizer state not loaded.")

    # scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"]

    print(f"✅ Model loaded from {PATH}")
    print(f"Checkpoint epoch: {start_epoch}")

elif response == "ABORT":
    print("🛑 Load aborted.")

else:
    print(f"❌ Invalid response '{response}'. Nothing was loaded.")


Checkpoint path:
../../../../../../projects/academic/sangst/zahidhas/Model_Weights/PanDerm/PanDerm_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth

Type LOAD to load the model, or ABORT to cancel:  LOAD


/scratch/26550277/ipykernel_3650865/433582041.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH, map_location=device)


Missing: []
Unexpected: []



Type OPTIMIZER to also load optimizer state, or SKIP to keep the current optimizer:  OPTIMIZER


✅ Optimizer state loaded.
✅ Model loaded from ../../../../../../projects/academic/sangst/zahidhas/Model_Weights/PanDerm/PanDerm_hyper_sarkar_head_64_Derm12345_weight_ce_multilayer_small_radius_2.pth
Checkpoint epoch: 0


In [12]:
model.logit_scale = nn.Parameter(torch.tensor(2.0).log(), requires_grad = False) 
model.logit_scale 

Parameter containing:
tensor(0.6931)